In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

In [ ]:
!ls ../artifacts/datasets/

In [ ]:
results_df = pd.read_parquet('../output/experiment_results.parquet')
results_df.sort_values('wape').head(10)

# Data

In [ ]:
LEVEL = 'level_12_weekly_item_store'

In [ ]:
df = pd.read_parquet(f'../artifacts/datasets/dataset_{LEVEL}.parquet')

df = df.sort_values("date").copy()
df["date"] = pd.to_datetime(df["date"])

df.head()

In [ ]:
TARGET = "sales"

ID_COLS = ["agg_id", "date"]

FEATURES = [c for c in df.columns if c not in ID_COLS + [TARGET]]

CATEGORICAL_FEATURES = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
]

NUMERICAL_FEATURES = [c for c in FEATURES if c not in CATEGORICAL_FEATURES]

FEATURES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

In [ ]:
VALID_MONTHS = 18
TEST_MONTHS = 6

first_date = df["date"].min()
last_date = df["date"].max()

test_start = last_date - pd.DateOffset(months=TEST_MONTHS)
valid_start = test_start - pd.DateOffset(months=VALID_MONTHS)

train = df[df["date"] < valid_start]
valid = df[(df["date"] >= valid_start) & (df["date"] < test_start)]
test = df[df["date"] >= test_start]

print(f"Dataset starts : {first_date:%Y-%m-%d} ({len(df):,} rows)")
print(f"Validation start : {valid_start:%Y-%m-%d} ({len(train):,} rows)")
print(f"Test start       : {test_start:%Y-%m-%d} ({len(valid):,} rows)")
print(f"Dataset ends   : {last_date:%Y-%m-%d} ({len(test):,} rows)")

X_train = train[FEATURES].copy()
y_train = train[TARGET]

X_valid = valid[FEATURES].copy()
y_valid = valid[TARGET]

X_test = test[FEATURES].copy()
y_test = test[TARGET]

for X in (X_train, X_valid, X_test):
    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].astype("category")

# Modelo simple

In [ ]:
from lightgbm import LGBMRegressor
import lightgbm as lgb
from src.evaluation.metrics import compute_wrmsse, wape

def wape_metric(y_true, y_pred):
    score = wape(y_true, y_pred)
    return "wape", score, False  # menor es mejor

def wrmsse_metric(y_true, y_pred):
    score = compute_wrmsse(train, valid, y_pred)
    return "wrmsse", score, False

In [ ]:
model = LGBMRegressor(
    objective="l1",
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric=[wrmsse_metric,],
    callbacks=[
        lgb.early_stopping(25, first_metric_only=True),
        lgb.log_evaluation(5),
    ],
)

In [ ]:
y_pred = model.predict(X_test)

wape_test = wape(y_test, y_pred)
wrmsse_test = compute_wrmsse(train, test, y_pred)

print(f"Test WAPE: {wape_test:.2%}")
print(f"Test WRMSSE: {wrmsse_test:.4f}")

# Modelo optimizado

In [ ]:
import optuna
from optuna.integration import LightGBMPruningCallback
import lightgbm as lgb
import numpy as np

In [ ]:
import lightgbm as lgb
import numpy as np
import optuna
from optuna.integration import LightGBMPruningCallback


cat_cols = CATEGORICAL_FEATURES.copy()

In [ ]:
def objective(trial):

    model = lgb.LGBMRegressor(
        objective="tweedie",
        tweedie_variance_power=trial.suggest_float("tweedie_variance_power", 1.1, 1.4),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        n_estimators=3000,
        num_leaves=trial.suggest_int("num_leaves", 31, 512),
        max_depth=trial.suggest_int("max_depth", 5, 15),
        min_child_samples=trial.suggest_int("min_child_samples", 20, 300),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        subsample_freq=1,
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-4, 10, log=True),
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric=wrmsse_metric,
        categorical_feature=cat_cols,
        callbacks=[
            lgb.early_stopping(50, first_metric_only=True, verbose=False),
            LightGBMPruningCallback(trial, "wrmsse"),
        ],
    )

    y_pred = model.predict(X_valid, num_iteration=model.best_iteration_)
    
    _, final_wrmsse, _ = wrmsse_metric(y_valid, y_pred)
    
    return final_wrmsse

In [ ]:
import os

os.makedirs("../artifacts", exist_ok=True)

study = optuna.create_study(
    study_name=f"study_{LEVEL}", 
    direction="minimize",
    storage="sqlite:///../artifacts/optuna_study.db",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(
        seed=42,
        multivariate=True,
        group=True,
    ),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
)

study.optimize(
    objective,
    n_trials=200,
    timeout=300,
    show_progress_bar=True,
)

best_trial = study.best_trial
print(f"Mejor WRMSSE: {best_trial.value:.4f}")
print("Mejores hiperparámetros:", best_trial.params)

# Modelo final

In [ ]:
model_params = {
    k: v for k, v in best_trial.params.items()
    if not k.startswith("use_")
}

# with open("best_params.json", "w") as f:
#     json.dump(model_params, f, indent=2)

final_model = lgb.LGBMRegressor(
    objective="l1",
    n_estimators=5000,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
    subsample_freq=1,
    **model_params,
)

final_model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric=wrmsse_metric,
    categorical_feature=cat_cols,
    callbacks=[
        lgb.early_stopping(50, first_metric_only=True),
        lgb.log_evaluation(0),
    ],
)

final_model.booster_.save_model("final_model.txt")

In [ ]:
y_pred = final_model.predict(X_test, num_iteration=final_model.best_iteration_)

wape_test = wape(y_test, y_pred)
wrmsse_test = compute_wrmsse(train, test, y_pred)

print(f"Test WAPE: {wape_test:.2%}")
print(f"Test WRMSSE: {wrmsse_test:.4f}")

# Explora predicciones

In [ ]:
df_pred = pd.DataFrame({
    "agg_id": test["agg_id"],
    "date": test["date"],
    "y_true": y_test.astype(int),
    "y_pred": y_pred.round().astype(int),
})

df_pred['error'] = df_pred['y_true'] - df_pred['y_pred']

In [ ]:
df_pred.query('agg_id == "FOODS_3_FOODS_CA_4_CA"').plot(x="date", y=["y_true", "y_pred"], figsize=(12, 6))